In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DilatedConvBlock(nn.Module):
    """
    A single dilated convolution block:
      Conv1d -> BatchNorm -> ReLU
    Uses padding so that sequence length is preserved ("same" length).
    """

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.conv = nn.Conv1d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )
        self.bn = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class ChromBPNetStyleMultiTaskAdditiveGene(nn.Module):
    """
    ChromBPNet-inspired multi-task model with:
      - Context-conditioned gene head: gene_FC = g_context(ctx) + g_seq(seq_features)
      - ATAC head: scalar fold-change per peak, uses both seq + context.

    Inputs
    ------
    seq_onehot : (batch, 4, seq_len)
        One-hot DNA sequence.
    tf_masks   : optional (batch, num_tf_masks, seq_len)
        Extra per-base channels, e.g. TF binding masks.
    context_ids: (batch,)
        Integer IDs for perturbation / drug_dosage conditions.

    Outputs
    -------
    gene_fold_change : (batch, num_genes)
    atac_fold_change : (batch, 1)
    """

    def __init__(
        self,
        seq_len: int = 500,
        num_genes: int = 18000,
        num_contexts: int = 100,    # number of distinct perturbation contexts
        num_tf_masks: int = 0,
        num_channels: int = 512,
        num_dilated_layers: int = 8,
        first_kernel_size: int = 21,
        dilated_kernel_size: int = 3,
        context_dim: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.num_genes = num_genes
        self.num_contexts = num_contexts
        self.num_tf_masks = num_tf_masks
        self.num_channels = num_channels
        self.context_dim = context_dim

        in_channels = 4 + num_tf_masks  # A,C,G,T + optional masks

        # --- First conv layer (ChromBPNet-style) ---
        first_padding = (first_kernel_size - 1) // 2
        self.first_conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=num_channels,
            kernel_size=first_kernel_size,
            padding=first_padding,
        )
        self.first_bn = nn.BatchNorm1d(num_channels)
        self.first_relu = nn.ReLU(inplace=True)

        # --- Dilated convolutional trunk ---
        dilated_layers = []
        for i in range(num_dilated_layers):
            dilation = 2 ** i  # 1, 2, 4, ..., 2^(L-1)
            dilated_layers.append(
                DilatedConvBlock(
                    channels=num_channels,
                    kernel_size=dilated_kernel_size,
                    dilation=dilation,
                )
            )
        self.dilated_trunk = nn.Sequential(*dilated_layers)

        # --- Context embedding ---
        self.context_embed = nn.Embedding(
            num_embeddings=num_contexts,
            embedding_dim=context_dim,
        )

        # --- Dropout ---
        self.dropout_seq = nn.Dropout(dropout)
        self.dropout_ctx = nn.Dropout(dropout)

        # --- Gene head: additive decomposition ---
        # gene_FC = fc_genes_context(ctx_repr) + fc_genes_seq(seq_repr)
        self.fc_genes_context = nn.Linear(context_dim, num_genes)
        self.fc_genes_seq     = nn.Linear(num_channels, num_genes)

        # --- ATAC head: uses both seq + context ---
        self.fc_atac = nn.Linear(num_channels + context_dim, 1)

    def forward(
        self,
        seq_onehot: torch.Tensor,
        tf_masks: torch.Tensor | None = None,
        context_ids: torch.Tensor | None = None,
    ):
        """
        seq_onehot : (B, 4, L)
        tf_masks   : (B, num_tf_masks, L) or None
        context_ids: (B,) LongTensor in [0, num_contexts-1]
        """
        # Basic checks
        if seq_onehot.dim() != 3:
            raise ValueError(
                f"seq_onehot must be (batch, 4, seq_len), got {seq_onehot.shape}"
            )
        if seq_onehot.size(1) != 4:
            raise ValueError(
                f"seq_onehot must have 4 channels for A,C,G,T, got {seq_onehot.size(1)}"
            )
        if seq_onehot.size(2) != self.seq_len:
            raise ValueError(
                f"Expected seq_len={self.seq_len}, got {seq_onehot.size(2)}"
            )

        if context_ids is None:
            raise ValueError("context_ids must be provided for conditioning.")
        if context_ids.dim() != 1:
            raise ValueError(
                f"context_ids must be (batch,), got shape {context_ids.shape}"
            )

        x = seq_onehot

        # Concatenate TF masks as extra channels if provided
        if tf_masks is not None:
            if tf_masks.dim() != 3:
                raise ValueError(
                    f"tf_masks must be (batch, num_tf_masks, seq_len), got {tf_masks.shape}"
                )
            if tf_masks.size(2) != self.seq_len:
                raise ValueError(
                    f"tf_masks seq_len mismatch: expected {self.seq_len}, got {tf_masks.size(2)}"
                )
            x = torch.cat([x, tf_masks], dim=1)  # (B, 4 + num_tf_masks, L)

        # --- Sequence trunk ---
        x = self.first_conv(x)
        x = self.first_bn(x)
        x = self.first_relu(x)

        x = self.dilated_trunk(x)          # (B, C, L)

        # Global average pooling over sequence
        seq_repr = x.mean(dim=2)           # (B, C)
        seq_repr = self.dropout_seq(seq_repr)

        # Context embedding
        ctx_repr = self.context_embed(context_ids)  # (B, context_dim)
        ctx_repr = self.dropout_ctx(ctx_repr)

        # --- Gene head: additive context + sequence ---
        gene_from_context = self.fc_genes_context(ctx_repr)  # (B, num_genes)
        gene_from_seq     = self.fc_genes_seq(seq_repr)      # (B, num_genes)
        gene_fold_change  = gene_from_context + gene_from_seq

        # --- ATAC head: uses both seq + context ---
        h_atac = torch.cat([seq_repr, ctx_repr], dim=1)      # (B, C + context_dim)
        atac_fold_change = self.fc_atac(h_atac)              # (B, 1)

        return gene_fold_change, atac_fold_change


In [ ]:
batch_size = 8
seq_len = 500
num_genes = 18000
num_tf_masks = 2
num_contexts = 64   # number of perturbation/drug_dosage contexts

model = ChromBPNetStyleMultiTaskAdditiveGene(
    seq_len=seq_len,
    num_genes=num_genes,
    num_contexts=num_contexts,
    num_tf_masks=num_tf_masks,
    context_dim=64,
)

# Dummy DNA input
bases_int = torch.randint(0, 4, (batch_size, seq_len))        # (B, L)
seq_onehot = F.one_hot(bases_int, num_classes=4).float()      # (B, L, 4)
seq_onehot = seq_onehot.permute(0, 2, 1)                      # (B, 4, L)

# Dummy TF masks
tf_masks = torch.randint(0, 2, (batch_size, num_tf_masks, seq_len)).float()

# Dummy context IDs
context_ids = torch.randint(0, num_contexts, (batch_size,), dtype=torch.long)

g_pred, atac_fc_pred = model(seq_onehot, tf_masks=tf_masks, context_ids=context_ids)
print(g_pred.shape)        # (8, 18000)
print(atac_fc_pred.shape)  # (8, 1)


In [ ]:
lambda_seq = 1e-4
seq_weights = model.fc_genes_seq.weight
reg_seq = lambda_seq * torch.sum(seq_weights ** 2)
loss = loss_genes + alpha * loss_atac + reg_seq
